In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("dataset.csv")

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

Dataset shape: (2123, 23)

First 5 rows:
   segment  anomaly  train   channel  sampling  duration  len          mean  \
0        1        1      1  CADC0872         1       279  280  8.533143e-07   
1        2        1      1  CADC0872         1       476  477 -3.639396e-06   
2        3        1      1  CADC0872         1       594  595  1.170788e-05   
3        4        1      1  CADC0872         1       271  272  8.486808e-07   
4        5        0      0  CADC0872         1       256  257  1.058485e-05   

            var       std  ...  smooth10_n_peaks  smooth20_n_peaks  \
0  3.494283e-10  0.000019  ...                 3                 2   
1  6.476485e-10  0.000025  ...                 1                 1   
2  5.592877e-10  0.000024  ...                 2                 2   
3  5.466024e-10  0.000023  ...                 2                 2   
4  5.279023e-10  0.000023  ...                 1                 1   

   diff_peaks  diff2_peaks      diff_var     diff2_var  gaps_sq

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDescribe (all columns):")
print(df.describe(include="all").T)

Column names:
['segment', 'anomaly', 'train', 'channel', 'sampling', 'duration', 'len', 'mean', 'var', 'std', 'kurtosis', 'skew', 'n_peaks', 'smooth10_n_peaks', 'smooth20_n_peaks', 'diff_peaks', 'diff2_peaks', 'diff_var', 'diff2_var', 'gaps_squared', 'len_weighted', 'var_div_duration', 'var_div_len']

Data types:
segment               int64
anomaly               int64
train                 int64
channel              object
sampling              int64
duration              int64
len                   int64
mean                float64
var                 float64
std                 float64
kurtosis            float64
skew                float64
n_peaks               int64
smooth10_n_peaks      int64
smooth20_n_peaks      int64
diff_peaks            int64
diff2_peaks           int64
diff_var            float64
diff2_var           float64
gaps_squared          int64
len_weighted          int64
var_div_duration    float64
var_div_len         float64
dtype: object

Missing values:
segment   

In [ ]:
# 1. Anomaly distribution
print("Anomaly distribution:")
print(df["anomaly"].value_counts())
print("\nPercentage:")
print(df["anomaly"].value_counts(normalize=True) * 100)

# 2. Channel analysis
print("\nNumber of unique channels:", df["channel"].nunique())
print("\nChannel distribution:")
print(df["channel"].value_counts())

# 3. Anomaly by channel
print("\nAnomaly distribution by channel:")
channel_anomalies = pd.crosstab(df["channel"], df["anomaly"])
print(channel_anomalies)

Anomaly distribution:
anomaly
0    1689
1     434
Name: count, dtype: int64

Percentage:
anomaly
0    79.55723
1    20.44277
Name: proportion, dtype: float64

Number of unique channels: 9

Channel distribution:
channel
CADC0873    593
CADC0872    546
CADC0888    252
CADC0892    211
CADC0874    194
CADC0884    158
CADC0894    144
CADC0890     14
CADC0886     11
Name: count, dtype: int64

Anomaly distribution by channel:
anomaly     0    1
channel           
CADC0872  415  131
CADC0873  488  105
CADC0874  125   69
CADC0884  158    0
CADC0886    8    3
CADC0888  192   60
CADC0890    3   11
CADC0892  177   34
CADC0894  123   21


In [ ]:

target = "anomaly"
excluded = ["segment", "anomaly", "train", "channel"]

features = [col for col in df.columns if col not in excluded]

print("Number of features:", len(features))
print("\nFeatures list:")
print(features)

Number of features: 19

Features list:
['sampling', 'duration', 'len', 'mean', 'var', 'std', 'kurtosis', 'skew', 'n_peaks', 'smooth10_n_peaks', 'smooth20_n_peaks', 'diff_peaks', 'diff2_peaks', 'diff_var', 'diff2_var', 'gaps_squared', 'len_weighted', 'var_div_duration', 'var_div_len']


In [ ]:
correlation = df[features + [target]].corr()[target].drop(target).abs().sort_values(ascending=False)

print("Top 10 features correlated with anomaly:")
print(correlation.head(10))


print("\nAll correlations:")
print(correlation)

Top 10 features correlated with anomaly:
smooth10_n_peaks    0.527219
len                 0.293134
sampling            0.267742
kurtosis            0.233905
var                 0.185838
std                 0.173805
n_peaks             0.172464
duration            0.172199
var_div_duration    0.163552
diff2_peaks         0.162844
Name: anomaly, dtype: float64

All correlations:
smooth10_n_peaks    0.527219
len                 0.293134
sampling            0.267742
kurtosis            0.233905
var                 0.185838
std                 0.173805
n_peaks             0.172464
duration            0.172199
var_div_duration    0.163552
diff2_peaks         0.162844
mean                0.161937
len_weighted        0.156882
diff_peaks          0.155948
diff2_var           0.153694
smooth20_n_peaks    0.132104
var_div_len         0.127807
gaps_squared        0.111426
diff_var            0.108702
skew                0.010606
Name: anomaly, dtype: float64


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

# 1. تحديد الميزات التي سنستخدمها (أفضل 10 من الارتباط)
selected_features = correlation.head(10).index.tolist()
print("Selected features:", selected_features)

# 2. فصل البيانات حسب train (الموجودة في الـ Dataset)
train_mask = df["train"] == 1
test_mask = df["train"] == 0

X_train = df.loc[train_mask, selected_features].values
y_train = df.loc[train_mask, "anomaly"].values

X_test = df.loc[test_mask, selected_features].values
y_test = df.loc[test_mask, "anomaly"].values

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# 3. توحيد البيانات (Scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. بناء Isolation Forest
model = IsolationForest(n_estimators=100, contamination=0.20, random_state=42)
model.fit(X_train_scaled)

# 5. التنبؤ (ملاحظة: IsolationForest يعطي -1 للشذوذ و 1 للطبيعي)
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# 6. تحويل النتائج إلى 0 و 1 (1 = شذوذ)
y_pred_train_binary = np.where(y_pred_train == -1, 1, 0)
y_pred_test_binary = np.where(y_pred_test == -1, 1, 0)

# 7. التقييم
print("\n📊 Training Results:")
print(f"Accuracy: {accuracy_score(y_train, y_pred_train_binary)*100:.2f}%")
print(f"F1 Score: {f1_score(y_train, y_pred_train_binary)*100:.2f}%")

print("\n📊 Testing Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test_binary)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, y_pred_test_binary)*100:.2f}%")

print("\n📋 Classification Report (Testing):")
print(classification_report(y_test, y_pred_test_binary))

print("\nConfusion Matrix (Testing):")
print(confusion_matrix(y_test, y_pred_test_binary))

Selected features: ['smooth10_n_peaks', 'len', 'sampling', 'kurtosis', 'var', 'std', 'n_peaks', 'duration', 'var_div_duration', 'diff2_peaks']

Training samples: 1594
Testing samples: 529

📊 Training Results:
Accuracy: 72.65%
F1 Score: 31.87%

📊 Testing Results:
Accuracy: 70.89%
F1 Score: 32.46%

📋 Classification Report (Testing):
              precision    recall  f1-score   support

           0       0.82      0.81      0.81       416
           1       0.32      0.33      0.32       113

    accuracy                           0.71       529
   macro avg       0.57      0.57      0.57       529
weighted avg       0.71      0.71      0.71       529


Confusion Matrix (Testing):
[[338  78]
 [ 76  37]]


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import numpy as np

# Use all features except segment, anomaly, train, channel
excluded = ["segment", "anomaly", "train", "channel"]
features = [col for col in df.columns if col not in excluded]

train_mask = df["train"] == 1
test_mask = df["train"] == 0

X_train = df.loc[train_mask, features].values
y_train = df.loc[train_mask, "anomaly"].values
X_test = df.loc[test_mask, features].values
y_test = df.loc[test_mask, "anomaly"].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model 1: IsolationForest with all features
model_iso = IsolationForest(n_estimators=200, contamination=0.20, random_state=42)
model_iso.fit(X_train_scaled)
y_pred_iso = np.where(model_iso.predict(X_test_scaled) == -1, 1, 0)

print("IsolationForest (All Features):")
print(f"Accuracy: {accuracy_score(y_test, y_pred_iso)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, y_pred_iso)*100:.2f}%")
print(classification_report(y_test, y_pred_iso))
print()

# Model 2: OneClassSVM
model_svm = OneClassSVM(nu=0.20, kernel='rbf', gamma='scale')
model_svm.fit(X_train_scaled)
y_pred_svm = np.where(model_svm.predict(X_test_scaled) == -1, 1, 0)

print("OneClassSVM (All Features):")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, y_pred_svm)*100:.2f}%")
print(classification_report(y_test, y_pred_svm))

IsolationForest (All Features):
Accuracy: 72.40%
F1 Score: 33.64%
              precision    recall  f1-score   support

           0       0.82      0.83      0.83       416
           1       0.35      0.33      0.34       113

    accuracy                           0.72       529
   macro avg       0.58      0.58      0.58       529
weighted avg       0.72      0.72      0.72       529


OneClassSVM (All Features):
Accuracy: 77.50%
F1 Score: 46.15%
              precision    recall  f1-score   support

           0       0.85      0.86      0.86       416
           1       0.47      0.45      0.46       113

    accuracy                           0.78       529
   macro avg       0.66      0.66      0.66       529
weighted avg       0.77      0.78      0.77       529



In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

features = [col for col in df.columns if col not in ["segment", "anomaly", "train", "channel"]]
train_mask = df["train"] == 1
test_mask = df["train"] == 0

X_train = df.loc[train_mask, features].values
y_train = df.loc[train_mask, "anomaly"].values
X_test = df.loc[test_mask, features].values
y_test = df.loc[test_mask, "anomaly"].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = []
for nu in [0.15, 0.18, 0.20, 0.22, 0.25, 0.30]:
    model = OneClassSVM(nu=nu, kernel='rbf', gamma='scale')
    model.fit(X_train_scaled)
    y_pred = np.where(model.predict(X_test_scaled) == -1, 1, 0)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append((nu, acc, f1))
    print(f"nu={nu:.2f} | Accuracy: {acc*100:.2f}% | F1 Score: {f1*100:.2f}%")

best_nu = max(results, key=lambda x: x[2])
print(f"\nBest nu: {best_nu[0]} | F1 Score: {best_nu[2]*100:.2f}%")

nu=0.15 | Accuracy: 77.88% | F1 Score: 40.61%
nu=0.18 | Accuracy: 77.50% | F1 Score: 44.65%
nu=0.20 | Accuracy: 77.50% | F1 Score: 46.15%
nu=0.22 | Accuracy: 76.94% | F1 Score: 46.96%
nu=0.25 | Accuracy: 74.48% | F1 Score: 46.64%
nu=0.30 | Accuracy: 70.89% | F1 Score: 44.60%

Best nu: 0.22 | F1 Score: 46.96%


In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

# Best model configuration
best_nu = 0.22

features = [col for col in df.columns if col not in ["segment", "anomaly", "train", "channel"]]
train_mask = df["train"] == 1
test_mask = df["train"] == 0

X_train = df.loc[train_mask, features].values
y_train = df.loc[train_mask, "anomaly"].values
X_test = df.loc[test_mask, features].values
y_test = df.loc[test_mask, "anomaly"].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train final model
final_model = OneClassSVM(nu=best_nu, kernel='rbf', gamma='scale')
final_model.fit(X_train_scaled)

# Predict on test data
y_pred_test = np.where(final_model.predict(X_test_scaled) == -1, 1, 0)

# Get indices of test data
test_indices = df[test_mask].index

# Create a dataframe with test results
test_results = df.loc[test_mask, ["segment", "channel", "anomaly", "train"]].copy()
test_results["predicted_anomaly"] = y_pred_test
test_results["is_correct"] = test_results["anomaly"] == test_results["predicted_anomaly"]

print("🔍 Test Results Summary:")
print(f"Total test samples: {len(test_results)}")
print(f"True anomalies: {test_results['anomaly'].sum()}")
print(f"Predicted anomalies: {test_results['predicted_anomaly'].sum()}")
print(f"Correct predictions: {test_results['is_correct'].sum()}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, y_pred_test)*100:.2f}%")

# Show some sample results
print("\n📊 Sample Predictions (Test Data):")
print(test_results.head(10))

🔍 Test Results Summary:
Total test samples: 529
True anomalies: 113
Predicted anomalies: 117
Correct predictions: 407
Accuracy: 76.94%
F1 Score: 46.96%

📊 Sample Predictions (Test Data):
    segment   channel  anomaly  train  predicted_anomaly  is_correct
4         5  CADC0872        0      0                  0        True
6         7  CADC0872        0      0                  0        True
11       12  CADC0872        0      0                  0        True
12       13  CADC0872        1      0                  0       False
17       18  CADC0872        0      0                  0        True
19       20  CADC0872        1      0                  0       False
20       21  CADC0872        1      0                  0       False
27       28  CADC0872        0      0                  0        True
29       30  CADC0872        0      0                  0        True
30       31  CADC0872        1      0                  0       False


In [ ]:
# عرض مثال حقيقي: حالة شاذة تم اكتشافها
false_negatives = test_results[(test_results["anomaly"] == 1) & (test_results["predicted_anomaly"] == 0)]
false_positives = test_results[(test_results["anomaly"] == 0) & (test_results["predicted_anomaly"] == 1)]

print("🔴 False Negatives (يجب أن تكون شاذة لكن النموذج قال طبيعية):")
print(f"Count: {len(false_negatives)}")
print(false_negatives.head(5))

print("\n🟢 False Positives (قال شاذة لكنها طبيعية):")
print(f"Count: {len(false_positives)}")
print(false_positives.head(5))

🔴 False Negatives (يجب أن تكون شاذة لكن النموذج قال طبيعية):
Count: 59
    segment   channel  anomaly  train  predicted_anomaly  is_correct
12       13  CADC0872        1      0                  0       False
19       20  CADC0872        1      0                  0       False
20       21  CADC0872        1      0                  0       False
30       31  CADC0872        1      0                  0       False
80       81  CADC0872        1      0                  0       False

🟢 False Positives (قال شاذة لكنها طبيعية):
Count: 63
     segment   channel  anomaly  train  predicted_anomaly  is_correct
217      218  CADC0872        0      0                  1       False
220      221  CADC0892        0      0                  1       False
223      224  CADC0892        0      0                  1       False
228      229  CADC0892        0      0                  1       False
265      266  CADC0874        0      0                  1       False


In [ ]:
# اختيار حالة شاذة اكتشفها النموذج (مثال)
detected_anomaly = test_results[test_results["predicted_anomaly"] == 1].iloc[0]

print("🔍 MIRA Investigation Report")
print("=" * 50)
print(f"Segment: {detected_anomaly['segment']}")
print(f"Channel: {detected_anomaly['channel']}")
print(f"True Label: {'Anomaly' if detected_anomaly['anomaly'] == 1 else 'Normal'}")
print(f"MIRA Prediction: {'ANOMALY' if detected_anomaly['predicted_anomaly'] == 1 else 'Normal'}")

# عرض القيم الفعلية للميزات لهذه النقطة
print("\n📊 Feature Values for this Segment:")
for feature in features:
    value = df.loc[detected_anomaly['segment'] - 1, feature]  # segment starts from 1
    print(f"  {feature}: {value:.6f}")

🔍 MIRA Investigation Report
Segment: 104
Channel: CADC0872
True Label: Anomaly
MIRA Prediction: ANOMALY

📊 Feature Values for this Segment:
  sampling: 1.000000
  duration: 174.000000
  len: 175.000000
  mean: -0.000000
  var: 0.000000
  std: 0.000012
  kurtosis: -1.345047
  skew: 0.056870
  n_peaks: 2.000000
  smooth10_n_peaks: 2.000000
  smooth20_n_peaks: 2.000000
  diff_peaks: 2.000000
  diff2_peaks: 5.000000
  diff_var: 0.000000
  diff2_var: 0.000000
  gaps_squared: 174.000000
  len_weighted: 175.000000
  var_div_duration: 0.000000
  var_div_len: 0.000000


In [ ]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 107.9 MB/s eta 0:00:00
